# Attack Graph Build — Generalised Attack Profile (GAP)

**Date:** 2026-04-08 (viewer v0.4: 2026-04-15)
**Author:** Marc Labouchardiere

This notebook implements the **Attack Graph Design Schema** (see `2026-04-08_MTDSim_AttackGraphDesignSchema.md`). It builds the **Generalised Attack Profile (GAP)** — a single, global technique-dependency graph derived from MITRE ATT&CK — using:

1. **Phase 1 — STIX parsing.** Extract techniques (collapsed to parent), tactics, platforms, and group/software/campaign provenance from `enterprise-attack.json`.
1b. **Phase 1b — Enrichment.** Attach motivation/region/alias data to each MITRE intrusion-set from vendored MISP galaxy + manual overrides (v0.4).
2. **Phase 2 — Co-occurrence mining.** FP-Growth association rule mining over the binary group + software usage matrix, with directionality assigned via canonical tactic ordering (Rahman et al., 2022). Retained but de-emphasised — see note below.
3. **Phase 3 — Supplementary edges.** Import the MITRE CTID **Attack Flow** corpus (`.afb` files) as the primary edge source, and run keyword-based ontology extraction over STIX descriptions.
4. **Phase 4/5 — Merge, validate, and visualise.** Multi-source consensus merging, cycle breaking, and an interactive **Cytoscape.js** viewer (`MITRETechniqueDependencyVisualiser`) with motivation / campaign / group filtering.

> **v0.4 narrative change:** the research narrative rests on MITRE CTID Attack Flow edges (defensible, curated). Co-occurrence and ontology edges are still built for audit/comparison but default to hidden in the UI and must be opted in via the preset radio or evidence toggles.

All permanent code lives in [`src/mtdsim/attacker/gap/`](../src/mtdsim/attacker/gap/). This notebook is the orchestration + reporting layer.

## 1. Setup

In [1]:
from pathlib import Path
import json

from mtdsim.attacker.gap import (
    build_gap,
    parse_stix_bundle,
    build_usage_matrix,
    mine_cooccurrence_edges,
    import_attack_flow_corpus,
    extract_ontology_edges,
    enrich_group_profiles,
    MITRETechniqueDependencyVisualiser,
    TACTIC_ORDER,
    MOTIVATION_CATEGORIES,
)
from mtdsim.attacker.gap.enrichment import motivation as _motivation

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent
STIX_BUNDLE = NOTEBOOK_DIR / 'enterprise-attack.json'
ATTACK_FLOW_CORPUS = NOTEBOOK_DIR / 'attack-flow' / 'attack-flow-main' / 'corpus'
OUT_DIR = NOTEBOOK_DIR / 'gap_out'                 # gitignored — local only
CANONICAL_DIR = REPO_ROOT / 'data' / 'gap'         # committed canonical JSON
OUT_DIR.mkdir(exist_ok=True)
CANONICAL_DIR.mkdir(parents=True, exist_ok=True)

print('STIX bundle:', STIX_BUNDLE.exists(), STIX_BUNDLE)
print('Attack Flow corpus:', ATTACK_FLOW_CORPUS.exists(), ATTACK_FLOW_CORPUS)
print('Local output dir:', OUT_DIR)
print('Canonical artifact dir:', CANONICAL_DIR)

STIX bundle: True /home/marc/GitHub/MTDSim/notebooks/enterprise-attack.json
Attack Flow corpus: True /home/marc/GitHub/MTDSim/notebooks/attack-flow/attack-flow-main/corpus
Local output dir: /home/marc/GitHub/MTDSim/notebooks/gap_out
Canonical artifact dir: /home/marc/GitHub/MTDSim/data/gap


## 2. Phase 1 — Parse STIX Bundle

Extract techniques, tactics, platforms, and provenance metadata. Sub-techniques collapse to parent (D7).

In [2]:
nodes, index = parse_stix_bundle(str(STIX_BUNDLE))

print(f'Techniques (parent-collapsed): {len(nodes)}')
print(f'Intrusion-sets:                {len(index["group_ids"])}')
print(f'Software (malware + tool):     {len(index["software_ids"])}')
print(f'Campaigns:                     {len(index["campaign_ids"])}')
print(f'`uses` relationships:          {len(index["relationships"])}')

# Spot-check a well-known technique
t1059 = nodes.get('T1059')
if t1059:
    print(f'\nExample: {t1059.technique_id} — {t1059.technique_name}')
    print(f'  primary tactic: {t1059.primary_tactic} (layer {t1059.tactic_layer})')
    print(f'  groups: {t1059.group_count}, campaigns: {t1059.campaign_count}')
    print(f'  sub-techniques collapsed: {len(t1059.sub_technique_ids)}')

Techniques (parent-collapsed): 216
Intrusion-sets:                172
Software (malware + tool):     784
Campaigns:                     52
`uses` relationships:          17270

Example: T1059 — Command and Scripting Interpreter
  primary tactic: execution (layer 3)
  groups: 120, campaigns: 38
  sub-techniques collapsed: 13


In [3]:
# Per-tactic technique counts (should mirror the ATT&CK matrix layout)
from collections import Counter
ctr = Counter(n.primary_tactic for n in nodes.values())
for tactic in TACTIC_ORDER:
    print(f'  {tactic:<22} {ctr.get(tactic, 0):>3}')

  reconnaissance          11
  resource-development     8
  initial-access          11
  execution               17
  persistence             20
  privilege-escalation     6
  defense-evasion         35
  credential-access       16
  discovery               31
  lateral-movement         6
  collection              15
  command-and-control     16
  exfiltration             9
  impact                  15


## 3. Phase 2 — Co-occurrence Mining (Primary Edges)

Build a binary `(entity x technique)` matrix from group + software usage and run FP-Growth association rule mining (Rahman et al., 2022). Directionality is assigned by canonical tactic ordering: rules where antecedent and consequent are in the same tactic layer are recorded as **intra-tactic unresolved** (Section 3.4) and dropped from the first iteration.

In [4]:
matrix = build_usage_matrix(nodes, index)
print(f'Usage matrix: {matrix.shape[0]} entities x {matrix.shape[1]} techniques')
print(f'Mean techniques per entity: {matrix.sum(axis=1).mean():.2f}')
print(f'Most-used techniques (top 10):')
top = matrix.sum(axis=0).sort_values(ascending=False).head(10)
for tid, count in top.items():
    print(f'  {tid:<10} {nodes[tid].technique_name[:50]:<50} {int(count)}')

Usage matrix: 952 entities x 216 techniques
Mean techniques per entity: 13.73
Most-used techniques (top 10):
  T1059      Command and Scripting Interpreter                  560
  T1105      Ingress Tool Transfer                              470
  T1027      Obfuscated Files or Information                    459
  T1071      Application Layer Protocol                         429
  T1082      System Information Discovery                       391
  T1083      File and Directory Discovery                       345
  T1070      Indicator Removal                                  339
  T1140      Deobfuscate/Decode Files or Information            312
  T1057      Process Discovery                                  295
  T1036      Masquerading                                       289


In [5]:
co_edges, intra_tactic, co_meta = mine_cooccurrence_edges(
    matrix, nodes,
    min_support=0.1,
    min_confidence=0.6,
)
print(f'Rules generated:        {co_meta["n_rules"]}')
print(f'Median confidence cut:  {co_meta["confidence_threshold"]:.3f}')
print(f'Rules after filter:     {co_meta["n_rules_filtered"]}')
print(f'Directed edges:         {len(co_edges)}')
print(f'Intra-tactic dropped:   {len(intra_tactic)}')

print('\nTop 10 highest-confidence co-occurrence edges:')
for e in sorted(co_edges, key=lambda x: -x.confidence)[:10]:
    print(f'  {e.source_id} -> {e.target_id}  conf={e.confidence:.3f} support={e.support:.3f} lift={e.lift:.2f}')

Rules generated:        115
Median confidence cut:  0.690
Rules after filter:     58
Directed edges:         42
Intra-tactic dropped:   15

Top 10 highest-confidence co-occurrence edges:
  T1566 -> T1204  conf=0.903 support=0.157 lift=4.34
  T1059 -> T1218  conf=0.827 support=0.136 lift=1.41
  T1566 -> T1059  conf=0.824 support=0.143 lift=1.40
  T1059 -> T1132  conf=0.820 support=0.120 lift=1.39
  T1059 -> T1562  conf=0.811 support=0.126 lift=1.38
  T1059 -> T1041  conf=0.807 support=0.153 lift=1.37
  T1071 -> T1041  conf=0.801 support=0.152 lift=1.78
  T1059 -> T1518  conf=0.774 support=0.129 lift=1.32
  T1033 -> T1105  conf=0.772 support=0.182 lift=1.56
  T1106 -> T1027  conf=0.768 support=0.167 lift=1.59


## 4. Phase 3 — Supplementary Edges

Two automated supplementary sources:
- **MITRE Attack Flow corpus** (`.afb` files): high-fidelity, manually curated technique sequences.
- **Ontology extraction**: simplified DeepOP-style keyword scan over STIX technique descriptions for precondition language.

In [6]:
af_edges = import_attack_flow_corpus(str(ATTACK_FLOW_CORPUS), nodes)
print(f'Attack Flow edges: {len(af_edges)}')
print('Sample:')
for e in af_edges[:5]:
    print(f'  {e.source_id} -> {e.target_id}  ({e.evidence[0].description[:80]})')

Attack Flow edges: 436
Sample:
  T1566 -> T1140  (Observed in 1 Attack Flow(s): Black Basta Ransomware)
  T1140 -> T1204  (Observed in 1 Attack Flow(s): Black Basta Ransomware)
  T1204 -> T1059  (Observed in 4 Attack Flow(s): Black Basta Ransomware, Hancitor DLL, Maastricht U)
  T1059 -> T1574  (Observed in 1 Attack Flow(s): Black Basta Ransomware)
  T1574 -> T1218  (Observed in 2 Attack Flow(s): Black Basta Ransomware, Cobalt Kitty Campaign)


In [7]:
ont_edges = extract_ontology_edges(nodes)
print(f'Ontology-extracted edges: {len(ont_edges)}')
print('Sample:')
for e in ont_edges[:5]:
    print(f'  {e.source_id} -> {e.target_id}')
    print(f'      {e.evidence[0].description[:120]}')

Ontology-extracted edges: 60
Sample:
  T1078 -> T1110
      Precondition language in T1110 description: 'for example, adversaries may attempt to brute force access to [valid accoun
  T1003 -> T1110
      Precondition language in T1110 description: 'for example, adversaries may attempt to brute force access to [valid accoun
  T1087 -> T1110
      Precondition language in T1110 description: 'for example, adversaries may attempt to brute force access to [valid accoun
  T1201 -> T1110
      Precondition language in T1110 description: 'for example, adversaries may attempt to brute force access to [valid accoun
  T1078 -> T1561
      Precondition language in T1561 description: 'to maximize impact on the target organization in operations where network-w


## 5. Phases 4 & 5 — Merge, Validate, Build the GAP

`build_gap` runs all phases end-to-end: it calls the parser, miner, importers, merges multi-source edges into consensus edges, breaks cycles by removing the weakest-evidence backward edge, and computes quality metrics.

In [8]:
gap, extras = build_gap(
    stix_bundle_path=str(STIX_BUNDLE),
    attack_flow_corpus_dir=str(ATTACK_FLOW_CORPUS),
    min_support=0.1,
    min_confidence=0.6,
    include_ontology=True,
    break_cycles=True,
)

viz = MITRETechniqueDependencyVisualiser(gap)
summary = viz.summary()
print(json.dumps(summary, indent=2))

{
  "version": "0.4",
  "total_techniques": 216,
  "techniques_with_edges": 142,
  "orphan_techniques": 74,
  "edge_count": 388,
  "consensus_edge_count": 18,
  "backward_edge_count": 117,
  "intra_tactic_unresolved": 15,
  "entry_nodes": 24,
  "objective_nodes": 30,
  "per_tactic_counts": {
    "privilege-escalation": 6,
    "execution": 17,
    "persistence": 20,
    "collection": 15,
    "lateral-movement": 6,
    "defense-evasion": 35,
    "credential-access": 16,
    "discovery": 31,
    "resource-development": 8,
    "command-and-control": 16,
    "reconnaissance": 11,
    "impact": 15,
    "initial-access": 11,
    "exfiltration": 9
  },
  "evidence_breakdown": {
    "ontology": 53,
    "attack_flow": 312,
    "co_occurrence": 42
  },
  "motivation_coverage": {
    "groups_total": 172,
    "groups_with_motivation": 65
  },
  "co_occurrence_params": {
    "min_support": 0.1,
    "min_confidence": 0.6,
    "median_threshold": 0.6897810218978102
  }
}


## 6. Quality Report

In [9]:
print('=== GAP Quality Report ===')
print(f'Total techniques:           {gap.total_techniques}')
print(f'Techniques with edges:      {gap.techniques_with_edges} '
      f'({gap.techniques_with_edges / gap.total_techniques:.1%})')
print(f'Orphan techniques:          {gap.orphan_techniques}')
print(f'Total edges:                {gap.edge_count}')
print(f'  Consensus edges (>=2 src) {gap.consensus_edge_count}')
print(f'  Backward edges (loops):   {gap.backward_edge_count}')
print(f'Entry nodes (in-deg 0):     {len(gap.entry_nodes)}')
print(f'Objective nodes (out 0):    {len(gap.objective_nodes)}')
print(f'Intra-tactic unresolved:    {gap.intra_tactic_unresolved}')

print('\nEdge evidence breakdown:')
for src_type, count in summary['evidence_breakdown'].items():
    print(f'  {src_type:<20} {count}')

print('\nFirst 10 consensus edges (>=2 independent sources):')
for e in sorted([e for e in gap.edges if e.source_count >= 2], key=lambda x: -x.source_count)[:10]:
    src_types = sorted({ev.source_type for ev in e.evidence})
    print(f'  {e.source_id} -> {e.target_id}  sources={src_types}  conf={e.confidence:.2f}')

print('\nFirst 10 orphan techniques (in-degree=0, out-degree=0):')
edge_tids = {e.source_id for e in gap.edges} | {e.target_id for e in gap.edges}
orphans = [n for tid, n in gap.nodes.items() if tid not in edge_tids]
for n in orphans[:10]:
    print(f'  {n.technique_id} {n.technique_name[:60]:<60} (tactic {n.primary_tactic})')

=== GAP Quality Report ===
Total techniques:           216
Techniques with edges:      142 (65.7%)
Orphan techniques:          74
Total edges:                388
  Consensus edges (>=2 src) 18
  Backward edges (loops):   117
Entry nodes (in-deg 0):     24
Objective nodes (out 0):    30
Intra-tactic unresolved:    15

Edge evidence breakdown:
  ontology             53
  attack_flow          312
  co_occurrence        42

First 10 consensus edges (>=2 independent sources):
  T1204 -> T1105  sources=['attack_flow', 'co_occurrence', 'ontology']  conf=1.00
  T1059 -> T1057  sources=['attack_flow', 'co_occurrence']  conf=1.00
  T1059 -> T1055  sources=['attack_flow', 'co_occurrence']  conf=1.00
  T1082 -> T1041  sources=['attack_flow', 'co_occurrence']  conf=1.00
  T1059 -> T1547  sources=['attack_flow', 'co_occurrence']  conf=1.00
  T1566 -> T1059  sources=['attack_flow', 'co_occurrence']  conf=1.00
  T1059 -> T1140  sources=['attack_flow', 'co_occurrence']  conf=1.00
  T1566 -> T1204  sour

## 7. Interactive Visualisation (v0.4 — Cytoscape.js)

The `MITRETechniqueDependencyVisualiser` now writes a self-contained HTML file powered by **Cytoscape.js + dagre**. Visual encoding:

- **Node colour** by tactic layer (14-colour palette); dashed border = orphan.
- **Node size** uniform (campaign-count size encoding removed per feedback — importance is exposed via the detail panel, not glyph area).
- **Edge colour** by primary evidence type; **width** ∝ confidence; **consensus** edges (≥2 sources) are thicker.
- **Solid** = forward (tactic-respecting); **dashed** = backward loop.

**UI surfaces (filters stack, all client-side):**
- *Preset*: `Attack Flow only (default)` / `Curated` / `All evidence` / `Custom`.
- *Evidence types* (chips): toggle any combination.
- *Motivation* (ETDA 4-cat): espionage / financial gain / financial crime / sabotage — filters via APT attribution.
- *APT group* and *Campaign* pick-lists — selecting campaigns dims everything not in those subgraphs.
- *Tactics*, full-text search, min-confidence / consensus / orphan toggles.
- *Advanced*: k-shortest-path queries (moved behind a disclosure — no longer default).

Side panels are collapsible and drag-resizable; legend is tabbed (Tactics / Evidence / Motivation / Line styles).

In [10]:
html_out = OUT_DIR / 'gap.html'
viz.render_interactive(str(html_out))
print(f'Wrote {html_out}  ({html_out.stat().st_size // 1024} KB)')
print('Open in a browser — all filtering is now in-UI, no need for multiple files.')

Wrote /home/marc/GitHub/MTDSim/notebooks/gap_out/gap.html  (629 KB)
Open in a browser — all filtering is now in-UI, no need for multiple files.


In [11]:
# Programmatic access to the payload — subgraph demos.
#
# Per the 2026-04-17 GAP Subgraphing Exploration (Part 10), motivation was
# removed from per-node / per-edge payload fields: it's a group-level
# attribute, not a technique attribute. `GroupProfile.motivations` is kept
# as metadata and as a validation overlay. The canonical subgraphing API
# is now the selector module.
from mtdsim.attacker.gap.viz import build_payload
from mtdsim.attacker.gap import TerminalObjectiveSelector, PlatformSelector

payload = build_payload(gap)

# Motivation coverage — still a group-level stat, still informative.
print('Motivation coverage:')
cov = _motivation.coverage_stats(gap.groups)
print(f'  {cov["with_motivation"]}/{cov["total"]} groups attributed '
      f'({cov["coverage"]:.1%})')
for cat, n in cov['by_motivation'].items():
    print(f'    {cat:<32} {n:>3} groups')

# Selector-based subgraphs: Strategy A (terminal-objective) + B (platform).
print('\nSelector subgraphs (recommended A + B):')
for sel in [
    TerminalObjectiveSelector(tactic='impact'),
    TerminalObjectiveSelector(tactic='exfiltration'),
    PlatformSelector(profile='Windows'),
    PlatformSelector(profile='Cloud'),
]:
    view = sel.select(gap)
    prov = view.provenance
    key = prov.get('tactic') or prov.get('technique') or prov.get('profile')
    print(f'  {prov["selector"]:<18} {key:<16}  '
          f'nodes={len(view.node_set):>3}  edges={len(view.edge_set):>3}')

# Example: build a campaign-restricted subgraph via the payload.
campaign_of_interest = 'AF:Black Basta Ransomware'
if campaign_of_interest in payload['campaigns']:
    tids = set(payload['campaigns'][campaign_of_interest]['technique_ids'])
    sub_edges = [e for e in payload['edges']
                 if campaign_of_interest.replace('AF:', '') in e['campaigns']]
    print(f'\nCampaign subgraph: {campaign_of_interest}')
    print(f'  techniques: {len(tids)}  edges: {len(sub_edges)}')
else:
    print(f'\nCampaign {campaign_of_interest!r} not found; pick from payload["campaigns"].')

Motivation coverage:
  65/172 groups attributed (37.8%)
    information_theft_espionage       60 groups
    financial_gain                     7 groups
    financial_crime                    5 groups
    sabotage_destruction               5 groups

Selector subgraphs (recommended A + B):
  TerminalObjective  impact            nodes=104  edges=318
  TerminalObjective  exfiltration      nodes=107  edges=316
  Platform           Windows           nodes= 26  edges= 13
  Platform           Cloud             nodes= 36  edges=  6

Campaign subgraph: AF:Black Basta Ransomware
  techniques: 33  edges: 25


## 8. Export GAP for Simulator Consumption

Section 10 deliverable #7: complete GAP exported as JSON in node-link format. The simulator's adversary module can load it and traverse the graph as a stochastic technique-selection model.

In [12]:
# Canonical artifact — committed under data/gap/ so reviewers can load the
# viewer without re-running the build pipeline.
canonical = CANONICAL_DIR / f'gap_v{gap.version}_latest.json'
gap.to_json(str(canonical))
print(f'Wrote {canonical.relative_to(REPO_ROOT)}  ({canonical.stat().st_size // 1024} KB)')

# Dated snapshot in local gap_out/ (gitignored) for build-to-build diffing
dated = OUT_DIR / f'gap_v{gap.version}_{gap.build_date}.json'
gap.to_json(str(dated))

# Sanity-check round-trip
from mtdsim.attacker.gap.schema import GeneralisedAttackProfile
with open(canonical) as f:
    loaded = GeneralisedAttackProfile.from_dict(json.load(f))
print(f'Round-trip OK: {loaded.total_techniques} techniques, '
      f'{loaded.edge_count} edges, {len(loaded.groups)} groups, '
      f'{len(loaded.campaigns)} campaigns')

Wrote data/gap/gap_v0.4_latest.json  (1128 KB)
Round-trip OK: 216 techniques, 388 edges, 172 groups, 87 campaigns


## 9. Limitations & Next Steps

Known limitations of this first iteration (mirroring Section 8 of the design schema):

- **Co-occurrence ≠ causation.** Co-occurrence rules capture observed co-usage, not proven dependency. Mitigated by lift filtering and consensus with curated sources.
- **Tactic ordering is an approximation.** It is used only for *direction*, not edge generation. Backward edges are kept but flagged.
- **Intra-tactic blind spot.** Co-occurring techniques in the same tactic layer have no directional heuristic and are dropped (count reported above).
- **Sparse coverage for rare techniques.** Techniques used by very few groups don't meet `min_support`. These show up as orphans and are the targets for Phase 4 manual annotation.
- **Attack Flow latch resolution is spatial.** The `.afb` save format wires edges via `generic_latch` objects with no back-reference to actions, so the importer matches latches to their closest action by `layout` position. This can mis-assign latches in unusually dense flows.

**Next steps (out of scope here):** intra-tactic resolution, technique clustering for `APTAttacker`, HMM-weighted edges, MTD blocker model, automated CTI extraction.